# Benchmark Results EDA

### Table of contents

1. [Data loading](#data-loading)
2. [Overview](#overview)
3. [F1 performance](#f1-performance)
   - [Overall accuracy](#overall-accuracy)
   - [Exact vs. fuzzy gap](#exact-vs-fuzzy-gap)
   - [Per-field breakdown](#per-field-breakdown)
4. [Score distributions](#score-distributions)
5. [Efficiency](#efficiency)
   - [Cost vs. accuracy](#cost-vs-accuracy)
   - [Latency](#latency)
6. [Per-document deltas](#per-document-deltas)
7. [Conclusions](#conclusions)

## Introduction

This notebook explores the benchmark results across 12 experiments — three providers (Claude, Gemini, GPT), two model tiers (lite, standard), and two extraction strategies (agentic multimodal, single-pass text). Each experiment was run on 83 NDA documents from the Kleister-NDA dev split.

In [73]:
import json
import math
from pathlib import Path

import altair as alt
import polars as pl

## Data loading


In [74]:
data_path = Path.cwd() / "data" / "results"

df_aggregated = pl.read_csv(data_path / "benchmark-aggregated.csv")
df_per_document = pl.read_csv(data_path / "benchmark-per-document.csv")

## Benchmark

### Top five

In [75]:
df_aggregated.select(
    pl.col(
        "provider",
        "tier",
        "strategy",
        "exact_f1_mean",
        "exact_f1_std",
        "latency_mean",
        "latency_p95",
        "cost_mean",
        "cost_std",
    )
).sort("exact_f1_mean", descending=True).head(5)

provider,tier,strategy,exact_f1_mean,exact_f1_std,latency_mean,latency_p95,cost_mean,cost_std
str,str,str,f64,f64,f64,f64,f64,f64
"""gemini""","""standard""","""agentic""",0.918359,0.125763,14.567474,31.293001,0.010635,0.006469
"""gemini""","""standard""","""single_pass""",0.915347,0.125566,9.789317,30.069521,0.00748,0.006002
"""gpt""","""standard""","""single_pass""",0.898883,0.154057,2.068752,2.992403,0.010944,0.004249
"""gemini""","""lite""","""single_pass""",0.89567,0.151544,1.413686,1.946596,0.001075,0.000449
"""gpt""","""standard""","""agentic""",0.895367,0.165382,8.452288,10.846992,0.024124,0.005973


## F1 score results

In [76]:
def make_provider_facet_plot(df: pl.DataFrame, metric: str) -> alt.Chart:
    bars = (
        alt.Chart()
        .mark_bar()
        .encode(
            x="strategy:N",
            y=alt.Y(f"mean({metric}):Q").title(f"Mean {metric}"),
            color="strategy:N",
            tooltip=[f"mean({metric}):Q"],
        )
    )

    error_bars = (
        alt.Chart()
        .mark_errorbar(extent="ci")
        .encode(x="strategy:N", y=alt.Y(f"mean({metric}):Q").title(f"Mean {metric}"))
    )

    return (
        alt.layer(bars, error_bars, data=df)
        .properties(width=150, height=200)
        .facet(column="provider:N", row="tier:N")
    )


make_provider_facet_plot(df_aggregated, "exact_f1_mean")

alt.FacetChart(...)

### Data export

In [77]:
records = df_aggregated.select(
    pl.col("provider", "tier", "strategy", "exact_f1_mean", "latency_mean", "cost_mean")
).to_dicts()

latency_max = max(r["latency_mean"] for r in records)
cost_max = max(r["cost_mean"] for r in records)


def make_tab_data(records, metric):
    return [
        {
            "tier": r["tier"],
            "provider": r["provider"],
            "strategy": r["strategy"],
            "value": r[metric],
        }
        for r in records
    ]


benchmark = {
    "series": [
        {"key": "agentic", "label": "Agentic", "color": "primary"},
        {"key": "single_pass", "label": "Single Pass", "color": "secondary"},
    ],
    "tabs": [
        {
            "id": "f1",
            "label": "F1 Score",
            "chartType": "faceted-bar",
            "unit": "%",
            "yDomain": [0, 1],
            "facet": {"rowKey": "tier", "colKey": "provider"},
            "xKey": "strategy",
            "yKey": "value",
            "data": make_tab_data(records, "exact_f1_mean"),
        },
        {
            "id": "latency",
            "label": "Latency",
            "chartType": "faceted-bar",
            "unit": "s",
            "yDomain": [0, math.ceil(latency_max)],
            "facet": {"rowKey": "tier", "colKey": "provider"},
            "xKey": "strategy",
            "yKey": "value",
            "data": make_tab_data(records, "latency_mean"),
        },
        {
            "id": "cost",
            "label": "Cost",
            "chartType": "faceted-bar",
            "unit": "$",
            "yDomain": [0, round(math.ceil(cost_max * 1000) / 1000, 3)],
            "facet": {"rowKey": "tier", "colKey": "provider"},
            "xKey": "strategy",
            "yKey": "value",
            "data": make_tab_data(records, "cost_mean"),
        },
    ],
}

with open(data_path / "blog" / "benchmark.json", "w") as f:
    json.dump(benchmark, f, indent=2)

## Per-field breakdown


In [78]:
def plot_field_heatmap(df: pl.DataFrame) -> alt.Chart:
    field_cols = [
        "exact_effective_date_f1_mean",
        "exact_party_f1_mean",
        "exact_jurisdiction_f1_mean",
        "exact_term_f1_mean",
    ]

    melted = (
        df.with_columns(
            pl.concat_str(
                [
                    pl.col("provider"),
                    pl.lit(" "),
                    pl.col("tier"),
                    pl.lit(" "),
                    pl.col("strategy"),
                ]
            ).alias("experiment")
        )
        .select(["experiment"] + field_cols)
        .unpivot(index="experiment", variable_name="column", value_name="f1")
        .with_columns(
            pl.col("column")
            .str.replace("exact_", "")
            .str.replace("_f1_mean", "")
            .alias("field")
        )
    )

    return (
        alt.Chart(melted.to_pandas())
        .mark_rect()
        .encode(
            x=alt.X(
                "field:N",
                title=None,
                sort=["effective_date", "party", "jurisdiction", "term"],
            ),
            y=alt.Y("experiment:N", title=None),
            color=alt.Color(
                "f1:Q",
                title="Exact F1",
                scale=alt.Scale(scheme="blues", domain=[0.5, 1.0]),
            ),
            tooltip=[
                alt.Tooltip("experiment:N"),
                alt.Tooltip("field:N"),
                alt.Tooltip("f1:Q", format=".3f", title="Exact F1"),
            ],
        )
        .properties(title="Per-field Exact F1", width=400, height=300)
    )


plot_field_heatmap(df_aggregated)

alt.Chart(...)

## Efficiency

### Cost vs. accuracy


In [79]:
def plot_cost_vs_f1(df: pl.DataFrame) -> alt.Chart:
    df_pd = df.with_columns(
        pl.concat_str(
            [
                pl.col("provider"),
                pl.lit(" "),
                pl.col("tier"),
                pl.lit(" "),
                pl.col("strategy"),
            ]
        ).alias("label")
    ).to_pandas()

    return (
        alt.Chart(df_pd)
        .mark_point(filled=True, size=100)
        .encode(
            x=alt.X(
                "cost_mean:Q",
                title="Mean cost per document (USD)",
                axis=alt.Axis(format="$.4f"),
            ),
            y=alt.Y(
                "exact_f1_mean:Q",
                title="Exact F1",
                scale=alt.Scale(domain=[0.6, 1.0]),
            ),
            color=alt.Color("provider:N", title="Provider"),
            shape=alt.Shape("tier:N", title="Tier"),
            tooltip=[
                alt.Tooltip("label:N", title="Experiment"),
                alt.Tooltip("exact_f1_mean:Q", format=".3f", title="Exact F1"),
                alt.Tooltip("cost_mean:Q", format="$.5f", title="Cost/doc"),
                alt.Tooltip("cost_total:Q", format="$.3f", title="Total cost"),
            ],
        )
        .properties(title="Cost vs. Accuracy Trade-off", width=400, height=300)
    )


plot_cost_vs_f1(df_aggregated)

alt.Chart(...)

### Latency


In [80]:
def plot_latency(df: pl.DataFrame) -> alt.LayerChart:
    df_pd = df.with_columns(
        pl.concat_str(
            [
                pl.col("provider"),
                pl.lit(" "),
                pl.col("tier"),
                pl.lit(" "),
                pl.col("strategy"),
            ]
        ).alias("label")
    ).to_pandas()

    bars = (
        alt.Chart(df_pd)
        .mark_bar()
        .encode(
            x=alt.X("latency_mean:Q", title="Latency (s)"),
            y=alt.Y("label:N", sort="-x", title=None),
            color=alt.Color("provider:N", title="Provider"),
            tooltip=[
                alt.Tooltip("label:N", title="Experiment"),
                alt.Tooltip("latency_mean:Q", format=".2f", title="Mean (s)"),
                alt.Tooltip("latency_median:Q", format=".2f", title="Median (s)"),
                alt.Tooltip("latency_p95:Q", format=".2f", title="P95 (s)"),
            ],
        )
    )

    p95 = (
        alt.Chart(df_pd)
        .mark_tick(color="black", thickness=2, size=15)
        .encode(
            x=alt.X("latency_p95:Q"),
            y=alt.Y("label:N", sort="-x"),
            tooltip=[alt.Tooltip("latency_p95:Q", format=".2f", title="P95 (s)")],
        )
    )

    return (bars + p95).properties(
        title="Mean Latency per Document (tick = P95)", width=400, height=320
    )


plot_latency(df_aggregated)

alt.LayerChart(...)

## Per-document deltas

In [81]:
def compute_deltas(df: pl.DataFrame) -> pl.DataFrame:
    agentic = df.filter(pl.col("strategy") == "agentic").select(
        "document_id", "provider", "tier", "exact_f1", "tokens"
    )

    single_pass = df.filter(pl.col("strategy") == "single_pass").select(
        "document_id", "provider", "tier", pl.col("exact_f1").alias("sp_f1")
    )

    return agentic.join(
        single_pass, on=["document_id", "provider", "tier"]
    ).with_columns(
        (pl.col("exact_f1") - pl.col("sp_f1")).alias("delta"),
        pl.col("tokens")
        .qcut(4, labels=["Short", "Medium", "Long", "Very long"], allow_duplicates=True)
        .alias("doc_length"),
    )


deltas = compute_deltas(df_per_document)

In [82]:
deltas.describe()

statistic,document_id,provider,tier,exact_f1,tokens,sp_f1,delta,doc_length
str,str,str,str,f64,f64,f64,f64,str
"""count""","""498""","""498""","""498""",498.0,498.0,498.0,498.0,"""498"""
"""null_count""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,"""0"""
"""mean""",null,null,null,0.821579,11074.138554,0.883963,-0.062385,null
"""std""",null,null,null,0.217853,7647.979471,0.172211,0.215939,null
"""min""","""028ea476-30fa-56da-98e4-f9bd77…","""claude""","""lite""",0.0,1545.0,0.0,-1.0,null
"""25%""",null,null,null,0.75,6505.0,0.75,-0.0833,null
"""50%""",null,null,null,0.875,9245.0,1.0,0.0,null
"""75%""",null,null,null,1.0,12289.0,1.0,0.0,null
"""max""","""ffcbd8dd-878a-57fd-914f-ee6a0c…","""gpt""","""standard""",1.0,84078.0,1.0,1.0,null


In [83]:
def plot_slopegraph(df: pl.DataFrame) -> alt.FacetChart:
    long = (
        df.select(["document_id", "provider", "tier", "exact_f1", "sp_f1", "delta"])
        .unpivot(
            on=["sp_f1", "exact_f1"],
            index=["document_id", "provider", "tier", "delta"],
            variable_name="strategy",
            value_name="f1",
        )
        .with_columns(
            pl.col("strategy").replace({"sp_f1": "Single-pass", "exact_f1": "Agentic"}),
            pl.when(pl.col("delta") > 0.001)
            .then(pl.lit("Improved"))
            .when(pl.col("delta") < -0.001)
            .then(pl.lit("Regressed"))
            .otherwise(pl.lit("Unchanged"))
            .alias("direction"),
        )
    )

    df_pd = long.to_pandas()
    strategy_order = ["Single-pass", "Agentic"]
    color_scale = alt.Scale(
        domain=["Improved", "Unchanged", "Regressed"],
        range=["#2ca02c", "#bbbbbb", "#d62728"],
    )
    opacity_scale = alt.Scale(
        domain=["Improved", "Unchanged", "Regressed"],
        range=[0.5, 0.15, 0.5],
    )

    lines = (
        alt.Chart()
        .mark_line(strokeWidth=1)
        .encode(
            x=alt.X(
                "strategy:N",
                sort=strategy_order,
                title=None,
                axis=alt.Axis(labelAngle=0),
            ),
            y=alt.Y("f1:Q", title="Exact F1", scale=alt.Scale(domain=[0, 1])),
            detail="document_id:N",
            color=alt.Color("direction:N", scale=color_scale, title="Direction"),
            opacity=alt.Opacity("direction:N", scale=opacity_scale, legend=None),
        )
    )

    points = (
        alt.Chart()
        .mark_point(filled=True, size=15)
        .encode(
            x=alt.X("strategy:N", sort=strategy_order),
            y=alt.Y("f1:Q"),
            color=alt.Color("direction:N", scale=color_scale),
            opacity=alt.Opacity(
                "direction:N",
                scale=alt.Scale(
                    domain=["Improved", "Unchanged", "Regressed"],
                    range=[0.6, 0.2, 0.6],
                ),
                legend=None,
            ),
            tooltip=[
                alt.Tooltip("provider:N"),
                alt.Tooltip("tier:N"),
                alt.Tooltip("strategy:N"),
                alt.Tooltip("f1:Q", format=".3f", title="Exact F1"),
                alt.Tooltip("delta:Q", format=".3f", title="F1 delta"),
                alt.Tooltip("direction:N"),
            ],
        )
    )

    return (
        alt.layer(lines, points, data=df_pd)
        .properties(width=130, height=250)
        .facet(
            column=alt.Column("provider:N", title="Provider"),
            row=alt.Row("tier:N", title="Tier"),
        )
        .resolve_scale(y="shared")
    )


plot_slopegraph(deltas)

alt.FacetChart(...)

In [84]:
def plot_delta_dotplot(df: pl.DataFrame) -> alt.FacetChart:
    df_ranked = df.with_columns(
        pl.when(pl.col("delta") > 0.001)
        .then(pl.lit("Improved"))
        .when(pl.col("delta") < -0.001)
        .then(pl.lit("Regressed"))
        .otherwise(pl.lit("Unchanged"))
        .alias("direction"),
    )

    df_agg = (
        df_ranked.group_by(["provider", "tier", "doc_length", "direction"])
        .agg(
            pl.col("delta").mean().alias("delta"),
            pl.len().alias("count"),
        )
    )

    df_pd = df_agg.to_pandas()
    color_scale = alt.Scale(
        domain=["Improved", "Unchanged", "Regressed"],
        range=["#2ca02c", "#bbbbbb", "#d62728"],
    )

    zero_line = (
        alt.Chart()
        .mark_rule(color="black", strokeDash=[4, 4], opacity=0.4)
        .encode(x=alt.datum(0))
    )

    dots = (
        alt.Chart()
        .mark_point(filled=True, opacity=0.7)
        .encode(
            x=alt.X(
                "delta:Q",
                title="Agentic \u2212 Single-pass F1",
                axis=alt.Axis(format="+.2f"),
            ),
            y=alt.Y(
                "doc_length:O",
                title="Doc length",
                sort=["Short", "Medium", "Long", "Very long"],
            ),
            color=alt.Color("direction:N", scale=color_scale, title="Direction"),
            size=alt.Size("count:Q", title="Observations"),
            tooltip=[
                alt.Tooltip("provider:N"),
                alt.Tooltip("tier:N"),
                alt.Tooltip("doc_length:O", title="Doc length"),
                alt.Tooltip("delta:Q", format="+.3f", title="Mean F1 delta"),
                alt.Tooltip("direction:N"),
                alt.Tooltip("count:Q", title="Observations"),
            ],
        )
    )

    return (
        alt.layer(zero_line, dots, data=df_pd)
        .properties(width=200, height=220)
        .facet(
            column=alt.Column("provider:N", title="Provider"),
            row=alt.Row("tier:N", title="Tier"),
        )
    )


plot_delta_dotplot(deltas)

alt.FacetChart(...)

## Conclusions